# RAG Pipeline — Course Materials Assistant
## 2.1 Load & Inspect

In [1]:
import os
from pathlib import Path
from pypdf import PdfReader

RAW_DIR = Path("../data/raw")
files = sorted(RAW_DIR.glob("*.pdf"))
print(f"Found {len(files)} PDF files")

docs = []  # list of dicts: {source, page_num, text}
failed = []

for f in files:
    try:
        reader = PdfReader(f)
        for i, page in enumerate(reader.pages):
            text = page.extract_text() or ""
            if text.strip():
                docs.append({"source": f.name, "page": i + 1, "text": text})
    except Exception as e:
        failed.append((f.name, str(e)))

print(f"Total pages loaded: {len(docs)}")
print(f"Failed files: {failed}")

Found 1 PDF files


fontTools is required to fully parse the encoding of a CFF Type1 font in font dictionary {'/BaseFont': '/WGDTRK+Helvetica', '/Encoding': IndirectObject(563, 0, 1876456695392), '/FirstChar': 23, '/FontDescriptor': IndirectObject(564, 0, 1876456695392), '/LastChar': 31, '/Subtype': '/Type1', '/Type': '/Font', '/Widths': [556, 556, 556, 556, 556, 556, 556, 556, 556]}, but is not installed. Consider installing fontTools if you encounter encoding problems.


Total pages loaded: 529
Failed files: []


In [2]:
assert all(isinstance(d["text"], str) and d["text"].strip() for d in docs), "Found bad text entries!"
print("All docs verified as clean, non-empty strings ✅")

All docs verified as clean, non-empty strings ✅


## 2.2 Chunking Strategy
Using fixed-size chunking with overlap: 800 characters per chunk, 120 character 
overlap (~15%). This balances retrieval precision (small enough chunks stay 
topically focused) against context completeness (overlap prevents cutting a 
concept in half at a chunk boundary). Word-boundary splitting avoids truncating 
mid-word.

In [3]:
def chunk_text(text, chunk_size=800, overlap=120):
    chunks = []
    start = 0
    length = len(text)
    
    while start < length:
        end = min(start + chunk_size, length)
        chunk = text[start:end]
        
        if end < length:
            last_space = chunk.rfind(" ")
            if last_space > chunk_size * 0.5:
                chunk = chunk[:last_space]
                end = start + last_space
        
        chunk = chunk.strip()
        if chunk:
            chunks.append(chunk)
        
        new_start = end - overlap
        if new_start <= start:
            new_start = start + max(1, chunk_size - overlap)
        start = new_start
    
    return chunks

In [4]:
all_chunks = []
chunk_id = 0
for d in docs:
    for c in chunk_text(d["text"]):
        all_chunks.append({
            "id": f"chunk_{chunk_id}",
            "source": d["source"],
            "page": d["page"],
            "text": c
        })
        chunk_id += 1

print(f"Total chunks: {len(all_chunks)}")
print(all_chunks[0])

Total chunks: 2313
{'id': 'chunk_0', 'source': 'AI Engineering.pdf', 'page': 1, 'text': 'Chip Huyen\n AI Engineering\nBuilding Applications  \nwith Foundation Models'}


In [5]:
from collections import Counter

source_counts = Counter(c["source"] for c in all_chunks)
print(source_counts)

Counter({'AI Engineering.pdf': 2313})


In [6]:
print(f"Number of docs loaded: {len(docs)}")
for d in docs:
    print(f"- {d['source']}: {len(d['text'])} characters")

Number of docs loaded: 529
- AI Engineering.pdf: 73 characters
- AI Engineering.pdf: 2330 characters
- AI Engineering.pdf: 1410 characters
- AI Engineering.pdf: 1144 characters
- AI Engineering.pdf: 70 characters
- AI Engineering.pdf: 1955 characters
- AI Engineering.pdf: 2999 characters
- AI Engineering.pdf: 4595 characters
- AI Engineering.pdf: 4574 characters
- AI Engineering.pdf: 4817 characters
- AI Engineering.pdf: 3328 characters
- AI Engineering.pdf: 1971 characters
- AI Engineering.pdf: 2527 characters
- AI Engineering.pdf: 2314 characters
- AI Engineering.pdf: 2625 characters
- AI Engineering.pdf: 2014 characters
- AI Engineering.pdf: 2753 characters
- AI Engineering.pdf: 2295 characters
- AI Engineering.pdf: 1515 characters
- AI Engineering.pdf: 1820 characters
- AI Engineering.pdf: 2540 characters
- AI Engineering.pdf: 2437 characters
- AI Engineering.pdf: 1715 characters
- AI Engineering.pdf: 2110 characters
- AI Engineering.pdf: 2056 characters
- AI Engineering.pdf: 2316 

In [7]:
none_chunks = [c for c in all_chunks if c["text"] is None]
print(f"Chunks with text=None: {len(none_chunks)}")

non_str_chunks = [c for c in all_chunks if not isinstance(c["text"], str)]
print(f"Non-string chunks: {len(non_str_chunks)}")
print(non_str_chunks[:3])

Chunks with text=None: 0
Non-string chunks: 0
[]


In [8]:
all_chunks = [c for c in all_chunks if isinstance(c["text"], str) and c["text"].strip()]
texts = [c["text"] for c in all_chunks]
print(f"Chunks after cleaning: {len(all_chunks)}")

Chunks after cleaning: 2313


## 2.3 Embeddings & Vector Store


In [9]:
from sentence_transformers import SentenceTransformer

embed_model = SentenceTransformer("all-MiniLM-L6-v2")

test = embed_model.encode(["hello world"])
print(test.shape)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

(1, 384)


In [10]:
import numpy as np

texts = [c["text"] for c in all_chunks]
print(f"Encoding {len(texts)} chunks...")

embeddings = embed_model.encode(
    texts,
    show_progress_bar=True,
    batch_size=64,
    convert_to_numpy=True
).astype("float32")

print(f"Embeddings shape: {embeddings.shape}")

Encoding 2313 chunks...


Batches:   0%|          | 0/37 [00:00<?, ?it/s]

Embeddings shape: (2313, 384)


In [11]:
import faiss

dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(embeddings)

print(f"Vectors in index: {index.ntotal}")

Vectors in index: 2313


In [12]:
import os
import json

VECTOR_STORE_DIR = "../data/vector_store"
os.makedirs(VECTOR_STORE_DIR, exist_ok=True)

faiss.write_index(index, os.path.join(VECTOR_STORE_DIR, "index.faiss"))

metadata = [
    {"id": c["id"], "source": c["source"], "page": c["page"], "text": c["text"]}
    for c in all_chunks
]
with open(os.path.join(VECTOR_STORE_DIR, "metadata.json"), "w", encoding="utf-8") as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

config = {
    "embedding_model": "all-MiniLM-L6-v2",
    "chunk_size": 800,
    "chunk_overlap": 120,
    "num_chunks": len(all_chunks)
}
with open(os.path.join(VECTOR_STORE_DIR, "config.json"), "w", encoding="utf-8") as f:
    json.dump(config, f, indent=2)

print("Saved index.faiss, metadata.json, config.json")

Saved index.faiss, metadata.json, config.json


In [13]:
index2 = faiss.read_index(os.path.join(VECTOR_STORE_DIR, "index.faiss"))
with open(os.path.join(VECTOR_STORE_DIR, "metadata.json"), encoding="utf-8") as f:
    metadata2 = json.load(f)

print(f"Reloaded index vectors: {index2.ntotal}")
print(f"Reloaded metadata entries: {len(metadata2)}")

Reloaded index vectors: 2313
Reloaded metadata entries: 2313


## 2.4 Retrieval & Prompting
Retrieval uses FAISS L2 nearest-neighbor search over the embedded chunks. 
Retrieved chunks are injected into a prompt template along with the user's 
question, and the model is instructed to answer only from the provided 
context and cite the source document + page.

In [14]:
def retrieve(query, k=4):
    query_vec = embed_model.encode([query], convert_to_numpy=True).astype("float32")
    distances, indices = index.search(query_vec, k)
    results = []
    for idx, dist in zip(indices[0], distances[0]):
        item = metadata[idx]
        results.append({**item, "distance": float(dist)})
    return results

# quick check
retrieve("what is fine-tuning?", k=3)

[{'id': 'chunk_1568',
  'source': 'AI Engineering.pdf',
  'page': 360,
  'text': 'backpropagation during the tuning process, allowing them to be\nadjusted for specific tasks.\n336 | Chapter 7: Finetuning',
  'distance': 0.7185713648796082},
 {'id': 'chunk_1571',
  'source': 'AI Engineering.pdf',
  'page': 361,
  'text': 'ning”, I searched for keywords\n“p_tuning” and “p tuning” to account for different spellings.\nFinetuning Techniques | 337',
  'distance': 0.8067211508750916},
 {'id': 'chunk_1551',
  'source': 'AI Engineering.pdf',
  'page': 356,
  'text': 'n’t a new concept, new types of models and fine‐\ntuning techniques have inspired many creative model-merging techniques, making\nthis section especially fun to write about.\nParameter-Efficient Finetuning\nIn the early days of finetuning, models were small enough that people could finetune\nentire models. This approach is called full finetuning. In full finetuning, the number\nof trainable parameters is exactly the same as the num

In [15]:
def build_prompt(question, retrieved_chunks):
    context = "\n\n".join(
        f"[Source: {c['source']}, page {c['page']}]\n{c['text']}"
        for c in retrieved_chunks
    )
    return f"""You are a helpful assistant answering questions based only on the provided context.
If the answer is not in the context, say you don't know — do not make anything up.
Cite the source document and page number for any claim you make.

Context:
{context}

Question: {question}

Answer (with citations):"""

In [16]:
import ollama

def generate_answer(question, k=4):
    retrieved = retrieve(question, k=k)
    prompt = build_prompt(question, retrieved)
    response = ollama.chat(
        model="llama3.2:3b",
        messages=[{"role": "user", "content": prompt}]
    )
    return response["message"]["content"], retrieved

In [17]:
answer, sources = generate_answer("What is fine-tuning in the context of large language models?")
print(answer)
print("\nSources used:")
for s in sources:
    print(f"- {s['source']} (page {s['page']})")

According to the context, fine-tuning in the context of large language models is a process where a model is trained on a smaller amount of data, often after having been pre-trained on a large amount of data.

As stated in the source document, "If you have a small amount of data, you might want to use PEFT methods on more advanced models. If you have a large amount of data, use full finetuning with smaller models." (Source: AI Engineering.pdf, page 395)

Additionally, the document mentions that "With 100 examples, more advanced models give much better performance after finetuning. With 550,000 examples, all models give similar performance after finetuning." (Source: AI Engineering.pdf, page 398)

It's also worth noting that the document explains that the choice of fine-tuning approach depends on the amount of data available, and that using full finetuning with smaller models is often preferred when there is a large amount of data.

Sources used:
- AI Engineering.pdf (page 32)
- AI Engin

## 2.6 Evaluation
Testing the pipeline on 10 representative questions about the "AI Engineering" 
book. Each is manually reviewed for whether the retrieved context was relevant 
and whether the answer was grounded or hallucinated.

In [18]:
test_questions = [
    "What is fine-tuning in the context of large language models?",
    "What is retrieval-augmented generation?",
    "What is prompt engineering?",
    "What are embeddings?",
    "What is the difference between fine-tuning and prompting?",
    "What is model evaluation used for in AI engineering?",
    "What are foundation models?",
    "What is inference in the context of LLMs?",
    "What is a system prompt?",
    "What is hallucination in language models?"
]

eval_results = []
for q in test_questions:
    answer, sources = generate_answer(q)
    eval_results.append({
        "question": q,
        "top_source": f"{sources[0]['source']} p.{sources[0]['page']}" if sources else "none",
        "answer": answer[:200] + ("..." if len(answer) > 200 else "")
    })
    print(f"Q: {q}")
    print(f"A: {answer[:200]}...")
    print("-" * 80)

Q: What is fine-tuning in the context of large language models?
A: According to the context, fine-tuning in the context of large language models refers to a process where a smaller model is trained on a larger dataset after initial training on a smaller dataset.

The...
--------------------------------------------------------------------------------
Q: What is retrieval-augmented generation?
A: According to the provided context, retrieval-augmented generation (RAG) is a solution for knowledge-intensive NLP tasks. It was coined in the paper "Retrieval-Augmented Gen‐eration for Knowledge-Inten...
--------------------------------------------------------------------------------
Q: What is prompt engineering?
A: According to the provided context, prompt engineering refers to the process of crafting an instruction that gets a model to generate the desired outcome.

[Source: AI Engineering.pdf, page 235]

This ...
-------------------------------------------------------------------------------

In [19]:
import pandas as pd

eval_df = pd.DataFrame(eval_results)
eval_df["correct"] = ""  
eval_df

,question,top_source,answer,correct
0,What is fine-tuning in the context of large la...,AI Engineering.pdf p.32,"According to the context, fine-tuning in the c...",
1,What is retrieval-augmented generation?,AI Engineering.pdf p.278,"According to the provided context, retrieval-a...",
2,What is prompt engineering?,AI Engineering.pdf p.235,"According to the provided context, prompt engi...",
3,What are embeddings?,AI Engineering.pdf p.158,"According to the provided context, embeddings ...",
4,What is the difference between fine-tuning and...,AI Engineering.pdf p.361,"According to the context, the difference betwe...",
5,What is model evaluation used for in AI engine...,AI Engineering.pdf p.183,"According to the context, model evaluation is ...",
6,What are foundation models?,AI Engineering.pdf p.124,I don't know the answer to this question based...,
7,What is inference in the context of LLMs?,AI Engineering.pdf p.454,In the context of LLMs (Large Language Models)...,
8,What is a system prompt?,AI Engineering.pdf p.241,A system prompt is a set of instructions provi...,
9,What is hallucination in language models?,AI Engineering.pdf p.131,"According to the provided context, hallucinati...",


In [20]:
eval_df.loc[0, "correct"] = "yes"
eval_df.loc[1, "correct"] = "yes"
eval_df.loc[2, "correct"] = "yes"
eval_df.loc[3, "correct"] = "yes"
eval_df.loc[4, "correct"] = "partial"
eval_df.loc[5, "correct"] = "yes"
eval_df.loc[6, "correct"] = "yes"
eval_df.loc[7, "correct"] = "yes"
eval_df.loc[8, "correct"] = "partial"
eval_df.loc[9, "correct"] = "yes"
eval_df

,question,top_source,answer,correct
0,What is fine-tuning in the context of large la...,AI Engineering.pdf p.32,"According to the context, fine-tuning in the c...",yes
1,What is retrieval-augmented generation?,AI Engineering.pdf p.278,"According to the provided context, retrieval-a...",yes
2,What is prompt engineering?,AI Engineering.pdf p.235,"According to the provided context, prompt engi...",yes
3,What are embeddings?,AI Engineering.pdf p.158,"According to the provided context, embeddings ...",yes
4,What is the difference between fine-tuning and...,AI Engineering.pdf p.361,"According to the context, the difference betwe...",partial
5,What is model evaluation used for in AI engine...,AI Engineering.pdf p.183,"According to the context, model evaluation is ...",yes
6,What are foundation models?,AI Engineering.pdf p.124,I don't know the answer to this question based...,yes
7,What is inference in the context of LLMs?,AI Engineering.pdf p.454,In the context of LLMs (Large Language Models)...,yes
8,What is a system prompt?,AI Engineering.pdf p.241,A system prompt is a set of instructions provi...,partial
9,What is hallucination in language models?,AI Engineering.pdf p.131,"According to the provided context, hallucinati...",yes


## 2.7 Export
The vector store, chunk metadata, and config were already persisted to disk 
in Section 2.3. Confirming the backend has everything it needs below.

In [21]:
print(os.listdir(VECTOR_STORE_DIR))

['chroma.sqlite3', 'config.json', 'index.faiss', 'metadata.json']
